# Second-Chern-form density $c_2(\mathbf k)$ behind a dθ/dQ

Replaces both `plot_c2_density.ipynb`. `calculations/run_axion.py` saves the density with every dθ:
`<mode run>/<sweep>/nk_<N>.npz` holds `c2_density` (nk, nk, nk, 2) for the base and mode ends of the
finite difference, and `dtheta = Σ_k c2 · d3k`. Older sweeps call it `chern2_density`; both are read.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))

In [ ]:
# a dθ sweep saved by run_axion.py (in the displaced run's folder)
SWEEP = DATA / "Y2Ir2O7/phonon/GM2-/mode1/Q_0.01A/output/178365bc889/trial_02_Y_p_Ir_d_O_sp/no_displacement/181565bc889/nk_sweep_mmn_shared_ws"
NK = 12
END = 0                                # 0: base end, 1: mode end
SLICES = 6                             # k3 planes to show

In [ ]:
folder = SWEEP
print("sweeps in that run:", sorted(p.name for p in SWEEP.parent.glob("nk_sweep*")))
data = np.load(folder / f"nk_{NK}.npz", allow_pickle=True)
c2_all = data["c2_density"] if "c2_density" in data.files else data["chern2_density"]
c2 = c2_all[..., END].real
d3k = float(data["d3k"]) if "d3k" in data.files else 1 / NK**3
print(f"{folder.relative_to(ROOT)}  nk={NK}  shape={c2.shape}")
print(f"dtheta saved = {np.asarray(data['dtheta'])[END].real:.6g},  sum(c2)*d3k = {c2.sum() * d3k:.6g}")

In [ ]:
planes = np.linspace(0, c2.shape[2] - 1, SLICES).round().astype(int)
v = np.abs(c2).max()
fig, axes = plt.subplots(1, SLICES, figsize=(3 * SLICES, 3.2), sharey=True)
for ax, k3 in zip(axes, planes):
    image = ax.imshow(c2[:, :, k3].T, origin="lower", extent=(0, 1, 0, 1), cmap="RdBu_r", vmin=-v, vmax=v)
    ax.set(title=f"$k_3$ = {k3}/{c2.shape[2]}", xlabel="$k_1$")
axes[0].set_ylabel("$k_2$")
fig.colorbar(image, ax=axes, shrink=0.8, label="$c_2$")
plt.show()

In [ ]:
# Where the integral comes from: cumulative |c2| weight versus plane
fig, ax = plt.subplots(figsize=(6, 3))
for axis, label in enumerate("123"):
    ax.plot(np.abs(c2).sum(axis=tuple(i for i in range(3) if i != axis)) * d3k, "o-", label=f"planes of $k_{label}$")
ax.set(xlabel="plane index", ylabel=r"$\sum |c_2|\,d^3k$")
ax.legend(frameon=False)
plt.show()